# Etapa II · Modelado predictivo GAC

Gestión de Proyectos. Análisis de correlaciones, regresión lineal simple y múltiple sobre conteos mensuales. Ejecuta desde la carpeta del proyecto con las dependencias de `requirements.txt`. Los datos usados son agregados y no contienen información de clientes.

## 1. Fuente y preparación

La fuente es la base original **Leads por Canal**, con el encabezado en la cuarta fila. Se toman los seis canales y las primeras 30 columnas mensuales; se excluyen totales, objetivos y proyecciones. Cada fila de `data/leads_mensuales.csv` representa un mes. Los años no aparecen en la fuente: se usan periodos consecutivos. Se conservan los ceros, incluidos los del primer mes; debe verificarse con el responsable si representan actividad nula o información no capturada.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from modelado import load_monthly, run_analysis, CHANNELS, DEFAULT_MULTIPLE

frame = load_monthly()
display(frame)
print("Meses de origen:", len(frame))

,Periodo,Mes,Piso,Plazas,Eventos,Planta,Digitales,Otros
0,1,Enero,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Febrero,329.0,0.0,7.0,6.0,6.0,195.0
2,3,Marzo,19.0,0.0,0.0,52.0,5.0,102.0
3,4,Abril,76.0,25.0,5.0,127.0,18.0,165.0
4,5,Mayo,87.0,28.0,0.0,60.0,69.0,178.0
5,6,Junio,55.0,68.0,71.0,376.0,87.0,310.0
6,7,Julio,92.0,294.0,4.0,237.0,113.0,214.0
7,8,Agosto,77.0,248.0,5.0,135.0,57.0,887.0
8,9,Septiembre,57.0,52.0,1.0,1030.0,107.0,68.0
9,10,Octubre,43.0,59.0,0.0,1214.0,139.0,518.0


Meses de origen: 30


## 2. Problema y separación temporal

Objetivo: predecir el conteo de **Digitales(t)** usando conteos del mes previo. Regresión simple: Digitales(t−1). Múltiple: Digitales(t−1), Plazas(t−1), Piso(t−1). Se pierde la primera observación al formar rezagos. El último 20% se reserva para prueba. No se mezclan meses al azar ni se ajusta con datos de prueba. Cada predicción de prueba utiliza el mes anterior real, ya observado: es una evaluación a un paso, no un pronóstico simultáneo de todo el horizonte.

In [2]:
target = "Digitales"
simple = "Digitales (t−1)"
multiple = DEFAULT_MULTIPLE
table, train, test, models, comparison, predictions = run_analysis(frame, target, simple, multiple, test_fraction=0.20)
print(f"Entrenamiento: {len(train)} meses, periodos {train.Periodo.min()}–{train.Periodo.max()}")
print(f"Prueba: {len(test)} meses, periodos {test.Periodo.min()}–{test.Periodo.max()}")
display(table.head())

Entrenamiento: 23 meses, periodos 2–24
Prueba: 6 meses, periodos 25–30


,Periodo,Mes,Digitales,Piso (t−1),Plazas (t−1),Eventos (t−1),Planta (t−1),Digitales (t−1),Otros (t−1)
0,2,Febrero,6.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,Marzo,5.0,329.0,0.0,7.0,6.0,6.0,195.0
2,4,Abril,18.0,19.0,0.0,0.0,52.0,5.0,102.0
3,5,Mayo,69.0,76.0,25.0,5.0,127.0,18.0,165.0
4,6,Junio,87.0,87.0,28.0,0.0,60.0,69.0,178.0


## 3. Correlaciones y boxplot

Pearson evalúa asociación lineal, Spearman asociación monótona mediante rangos y Kendall concordancia entre pares. Las correlaciones no prueban causalidad y pueden reflejar tendencias compartidas. Se exploran sólo los meses disponibles durante entrenamiento. El heatmap usa canales del mismo mes; la tabla de predictores usa X(t−1) frente a Y(t).

In [3]:
exploration = frame[frame.Periodo <= train.Periodo.max()]
for method in ["pearson", "spearman", "kendall"]:
    print(method.capitalize())
    display(exploration[CHANNELS].corr(method=method).round(3))
fig = px.imshow(exploration[CHANNELS].corr(), text_auto=".2f", zmin=-1, zmax=1,
                color_continuous_scale=["#236B8E", "white", "#D9082F"], title="Pearson · meses de entrenamiento")
fig.show()
lag_correlations = train[[target, *[f"{c} (t−1)" for c in CHANNELS]]].corr()[target].drop(target)
display(lag_correlations.rename("Correlación con Digitales(t)").to_frame())
long = exploration.melt(id_vars="Periodo", value_vars=CHANNELS, var_name="Canal", value_name="Leads")
px.box(long, x="Canal", y="Leads", points="all", color_discrete_sequence=["#D9082F"], title="Dispersión y valores extremos").show()

Pearson


,Piso,Plazas,Eventos,Planta,Digitales,Otros
Piso,1.000,-0.135,-0.002,-0.245,-0.111,0.094
Plazas,-0.135,1.000,0.218,-0.010,0.770,-0.114
Eventos,-0.002,0.218,1.000,-0.165,0.358,-0.063
Planta,-0.245,-0.010,-0.165,1.000,-0.024,0.103
Digitales,-0.111,0.770,0.358,-0.024,1.000,-0.197
Otros,0.094,-0.114,-0.063,0.103,-0.197,1.000


Spearman


,Piso,Plazas,Eventos,Planta,Digitales,Otros
Piso,1.000,0.185,0.385,-0.273,0.069,0.299
Plazas,0.185,1.000,0.368,0.322,0.822,-0.137
Eventos,0.385,0.368,1.000,0.072,0.418,-0.044
Planta,-0.273,0.322,0.072,1.000,0.374,-0.021
Digitales,0.069,0.822,0.418,0.374,1.000,-0.150
Otros,0.299,-0.137,-0.044,-0.021,-0.150,1.000


Kendall


,Piso,Plazas,Eventos,Planta,Digitales,Otros
Piso,1.000,0.132,0.296,-0.244,0.055,0.234
Plazas,0.132,1.000,0.246,0.230,0.652,-0.066
Eventos,0.296,0.246,1.000,0.015,0.283,-0.034
Planta,-0.244,0.230,0.015,1.000,0.239,-0.033
Digitales,0.055,0.652,0.283,0.239,1.000,-0.091
Otros,0.234,-0.066,-0.034,-0.033,-0.091,1.000


,Correlación con Digitales(t)
Piso (t−1),-0.127147
Plazas (t−1),0.815153
Eventos (t−1),0.298452
Planta (t−1),-0.086784
Digitales (t−1),0.803667
Otros (t−1),-0.260928


## 4. Regresión lineal simple

Modelo: ŷ(t) = β₀ + β₁ Digitales(t−1). Los coeficientes se estiman por mínimos cuadrados ordinarios con intercepto. Se superponen la dispersión real, las predicciones de prueba y la recta ajustada. El coeficiente expresa asociación en unidades de leads.

In [4]:
result = models["Simple"]
display(result["coefficients"])
fig = go.Figure()
fig.add_scatter(x=train[simple], y=train[target], mode="markers", name="Real · entrenamiento", marker_color="#87919E")
fig.add_scatter(x=test[simple], y=test[target], mode="markers", name="Real · prueba", marker_color="#236B8E")
fig.add_scatter(x=test[simple], y=result["test_prediction"], mode="markers", name="Predicho · prueba", marker=dict(color="#D9082F", symbol="x", size=11))
grid = np.linspace(table[simple].min(), table[simple].max(), 100)
fig.add_scatter(x=grid, y=result["model"].predict(pd.DataFrame({simple: grid})), mode="lines", name="Recta ajustada", line_color="#D9082F")
fig.update_layout(template="plotly_white", xaxis_title=simple, yaxis_title=target, title="Regresión simple: dispersión real y predicha")
fig.show()

,Variable,Coeficiente
0,Intercepto,45.816377
1,Digitales (t−1),0.823893


## 5. Regresión lineal múltiple

Modelo: ŷ(t) = β₀ + β₁ Digitales(t−1) + β₂ Plazas(t−1) + β₃ Piso(t−1). Los coeficientes se interpretan manteniendo constantes los otros predictores. La superposición por periodo permite comparar el resultado conjunto con la dispersión real; no se dibuja una recta bidimensional que pretenda representar tres predictores.

In [5]:
result = models["Múltiple"]
display(result["coefficients"])
print("Número de condición de X estandarizada:", round(result["condition"], 3))
for name, result in models.items():
    fig = go.Figure()
    fig.add_scatter(x=train.Periodo, y=train[target], mode="markers", name="Real · entrenamiento", marker_color="#87919E")
    fig.add_scatter(x=test.Periodo, y=test[target], mode="markers+lines", name="Real · prueba", marker_color="#236B8E")
    fig.add_scatter(x=test.Periodo, y=result["test_prediction"], mode="markers+lines", name="Predicho · prueba", marker=dict(color="#D9082F", symbol="x", size=11))
    fig.add_vline(x=test.Periodo.min() - 0.5, line_dash="dash")
    fig.update_layout(template="plotly_white", title=f"Regresión {name.lower()}: real y predicho superpuestos", xaxis_title="Periodo consecutivo", yaxis_title="Leads Digitales")
    fig.show()

,Variable,Coeficiente
0,Intercepto,2.631299
1,Digitales (t−1),0.445312
2,Plazas (t−1),0.533680
3,Piso (t−1),-0.019937


Número de condición de X estandarizada: 2.766


## 6. Evaluación y residuos

MAE es el error absoluto promedio; RMSE penaliza más los errores grandes. Ambos están en leads. R² puede ser negativo y no está definido si el objetivo es constante. La persistencia repite el mes anterior y sirve como referencia operativa. Todas las alternativas usan las mismas observaciones de prueba.

In [6]:
display(comparison.round(3))
display(predictions.round(3))
for name, result in models.items():
    fig = px.scatter(x=result["test_prediction"], y=test[target].to_numpy() - result["test_prediction"],
                     labels={"x":"Valor predicho", "y":"Real − predicho"}, title=f"Residuos de prueba · {name}", color_discrete_sequence=["#D9082F"])
    fig.add_hline(y=0, line_dash="dash")
    fig.show()
print("Menor MAE en esta partición:", comparison.loc[comparison.MAE.idxmin(), "Modelo"])

,Modelo,Predictores,MAE,RMSE,R²,R² entrenamiento
0,Simple,1,67.258,80.865,-0.701,0.646
1,Múltiple,3,63.878,69.717,-0.264,0.743
2,Persistencia,1,74.000,88.681,-1.046,NaN


,Periodo,Mes,Real,Simple,Residuo Simple,Múltiple,Residuo Múltiple,Persistencia
23,25,Enero,438.0,379.493,58.507,354.913,83.087,405.0
24,26,Febrero,338.0,406.682,-68.682,350.352,-12.352,438.0
25,27,Marzo,307.0,324.292,-17.292,382.356,-75.356,338.0
26,28,Abril,443.0,298.752,144.248,351.254,91.746,307.0
27,29,Mayo,427.0,410.801,16.199,385.651,41.349,443.0
28,30,Junio,299.0,397.619,-98.619,378.382,-79.382,427.0


Menor MAE en esta partición: Múltiple


## 7. Conclusiones de la configuración inicial

La regresión múltiple obtiene MAE de **63.88 leads** frente a **67.26** de la simple y **74.00** de persistencia. Los R² de prueba son **−0.264** y **−0.701**, respectivamente. Aunque la múltiple reduce el error, ambas regresiones muestran un ajuste débil en este bloque. R² negativo significa que el error cuadrático supera el de la referencia retrospectiva de predecir siempre la media del objetivo en prueba; esa media no es un pronóstico disponible de antemano.

Los 30 meses de origen son insuficientes para asegurar estabilidad. Deben estudiarse tendencia, estacionalidad, autocorrelación y cambios operativos. Los gráficos no comprueban por sí solos todos los supuestos de regresión. No se infiere causalidad ni significancia estadística. Se requieren más meses y una nueva evaluación fuera de muestra antes de usar el modelo para planificación. Estas cifras corresponden a la configuración inicial; si cambias variables, utiliza las tablas recalculadas para actualizar las conclusiones.

## 8. Dashboard y archivos de entrega

Ejecuta `python -m streamlit run app_univariado.py --server.port 8502`. El dashboard añade navegación por etapas, heatmap, boxplot, selectores, controles de partición, tablas, descargas y simulador. Los datos originales personales permanecen fuera de Git mediante `.gitignore`. Publicar el repositorio y entregar su enlace en Canvas son pasos separados.

### Referencias
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [Evaluación temporal](https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split)
- [R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)